In [ ]:
# ---------------------------------------------------------------------------
# STEP 3: Machine Learning Model Training & Evaluation
# ---------------------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import os

# 1. Load the processed dataset
input_file = 'processed_qsar_dataset.csv'

if not os.path.exists(input_file):
    raise FileNotFoundError("Dataset missing. Ensure Step 2 ran successfully!")

df = pd.read_csv(input_file)

# 2. Define Features (X) and Target (y)
# Drop the target variable and any text/metadata columns to isolate the numeric features
X = df.drop(columns=['molecule_chembl_id', 'canonical_smiles',
                     'standard_type', 'standard_value',
                     'standard_units', 'pIC50'])
y = df['pIC50']

print(f"Input Features Shape (X): {X.shape}")
print(f"Target Variable Shape (y): {y.shape}")

# 3. Train-Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining set size: {X_train.shape[0]} molecules")
print(f"Testing set size: {X_test.shape[0]} molecules")

# 4. Initialize and Train the Random Forest Model
print("\nTraining Random Forest Regressor... (This may take a minute)")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Make Predictions and Evaluate Model Performance
y_pred = model.predict(X_test)

# Calculate R-squared and Root Mean Square Error (RMSE)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n--- MODEL PERFORMANCE ---")
print(f"R-squared (R²): {r2:.3f}")
print(f"RMSE: {rmse:.3f}")

# Optional: A simple scatter plot of Experimental vs Predicted values
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--')
plt.title("Experimental vs Predicted pIC50")
plt.xlabel("Experimental pIC50")
plt.ylabel("Predicted pIC50")
plt.show()

# 6. Feature Importance Visualization
# Which molecular features drove the model's predictions?
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X.columns

# Get the top 20 most important features
top_n = 20
top_indices = indices[:top_n]
top_features = [feature_names[i] for i in top_indices]
top_importances = importances[top_indices]

# Plot the Feature Importances
plt.figure(figsize=(10, 6))
plt.title(f"Top {top_n} Most Important Molecular Features")
plt.bar(range(top_n), top_importances, align="center")
plt.xticks(range(top_n), top_features, rotation=45, ha='right')
plt.xlabel("Feature (Lipinski Descriptor or Morgan Bit)")
plt.ylabel("Relative Importance")
plt.tight_layout()
plt.show()